This script mostly has to do with the RAG and LLM pipeline originally theorized for this project. 
Due to CPU limitations, model sizes, and time constraints, they were not used in the final pipeline. A deterministic approach was used instead to adjust for this delayed inference time -- usability over complexity was prioritized. 

In [ ]:
import json
import faiss
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
tractian_docs = [
    "Tractian is a machine intelligence company that offers industrial monitoring systems.",
    "Tractian builds streamlined hardware-software solutions to give maintenance technicians and industrial decision-makers comprehensive oversight of their operations. ",
    "It is democratizing access to sophisticated real-time monitoring and asset operations tools.", # LinkedIn description line1 
    "The company’s broad market reach is evidenced in its customer base from various industries, such as John Deere, Procter & Gamble, Caterpillar, Goodyear, Carrier, Johnson Controls, and Bimbo, the owner of the brands Little Bites and Thomas Bagels.", # LinkedIn description line2
    "Tractian launched the AI-Assisted Maintenance category in the industrial sector. suggests preventive actions to be taken, giving invaluable insight and support to maintenance professionals.",
    "The intent of Assisted Maintenance is to help maintenance professionals to provide more assertive diagnosis with human-in-the-loop feedback.", # LinkedIn description line3
    "By combining shop floor expertise with our technology, maintainers will be able to anticipate and address issues with unprecedented accuracy and speed.", # LinkedIn description line4
    "Sensors stream real-time vibration, ultrasound temperature, and RPM. AI maps the signal patterns to specific failure modes (e.g., bearing wear, misalignment, unbalance). Your data is benchmarked against millions of historical machine hours and bearing libraries to make an accurate, explainable call, complete with evidence.",
    "Alerts are ranked by failure severity and asset criticality, so mission-critical machines rise to the top while low-impact noise stays out of the way. You get an alert that spells out the root cause and next steps. The only thing left to do is the fix.", # Failure detection on website 
    "What assets can Tractaian sensors monitor? Most rotating equipment across your plant -- motors, pumps, fans/blowers, gearboxes, compressors, conveyors, and more. If it spins and has bearings. Got something unusual -- slow-speed, vertical, submerged, or reciprocating? We will talk it through with you", # Asset types from website
    "Examples of companies that are a good fit for Tractian's Ideal Customer Profile (ICP) and Tractian has worked with include Georgia Aquarium, AirLiquide, ScottsMiracleGro, Ingredion, KraftHeinz, Whirlpool, CSX, Verizon, Kubota, Greif, Mauser Packaging Solutions, Hyundai, CAT, Carrier, Suzano, and DexCo" # companies worked with previously
]

In [ ]:
'''
# Embeddings with SentenceTransformer

embed_model = SentenceTransformer("avsolatorio/GIST-all-MiniLM-L6-v2")
doc_vectors = embed_model.encode(
    tractian_docs,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Build FAISS index
dimension = doc_vectors.shape[1]
index = faiss.IndexFlatIP(dimension)  # cosine similarity
index.add(doc_vectors)


# Retrieve top-k docs
def retrieve_docs(query, k=3):
    q_vector = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    distances, indices = index.search(q_vector, k)
    return [tractian_docs[i] for i in indices[0]]

# Load HuggingFace model (CPU)

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16  # faster, less memory
    
)

model.eval()
torch.set_grad_enabled(False)

'''


In [ ]:
'''
prompt = "Hello, I am a test for TinyLlama."
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
'''

In [ ]:
'''
def score_company(company_details, k=3, max_desc_length=50):
    # Shorten description for speed
    company_desc = company_details.get("description", "")[:max_desc_length]
    company_industry = ", ".join(company_details.get("industriesV2", []))
    parent_industry = ", ".join(company_details.get("industriesLegacy", []))
    specialties = ", ".join(company_details.get("specialties", []))
    staff_count = company_details.get("staffCount", "Unknown")
    num_locations = len(company_details["locations"])
    operational_footprint = min(1.0, (staff_count / 2000 + num_locations / 10) / 2)


    # Retrieve top-k docs from FAISS
    query = f"{company_industry} {parent_industry} {specialties} {company_desc}"
    context_docs = retrieve_docs(query, k=k)
    context_text = "\n".join(context_docs)
    
    # Build prompt: structured fields first, description after
    prompt = f"""
Use ONLY the following context about Tractian ICP to answer:

{context_text}

Structured information:
Industries: {company_industry}
Parent industries: {parent_industry}
Specialties: {specialties}
Staff count: {staff_count}

Company description (shortened):
{company_desc}

Score the company on these features from 0 to 1:

1. industrial_sector_fit
2. equipment_intensity
3. maintenance_operations
4. operational_footprint
5. cost_of_downtime
6. company_size

Return JSON only.
"""
    
    # Tokenize + generate
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract JSON safely
    try:
        start = response.find("{")
        end = response.rfind("}") + 1
        score_json = json.loads(response[start:end])
    except json.JSONDecodeError:
        score_json = {}
    score_json["operational_footprint"] = round(operational_footprint, 2)
    # Normalize scores 0–1
    for key in [
        "industrial_sector_fit",
        "equipment_intensity",
        "maintenance_operations",
        "operational_footprint",
        "cost_of_downtime",
        "company_size"
    ]:
        val = score_json.get(key, 0)
        try:
            score_json[key] = min(max(float(val), 0), 1)
        except:
            score_json[key] = 0
    
    return score_json

'''

In [ ]:
# Always predicts scores of zero
# Could do a deterministic approach instead 

# Use a regressor to predict scores based on structured fields and description embedding similarity to ICP docs

company_desc = company_details.get("description", "")[:150]
company_industry = ", ".join(company_details.get("industriesV2", []))
parent_industry = ", ".join(company_details.get("industriesLegacy", []))
specialties = ", ".join(company_details.get("specialties", []))
staff_count = company_details.get("staffCount", "Unknown")
num_locations = len(company_details.get("locations", []))

# Create feature vector
# Apple 
# Kalshi
# Kraft Heinz 
# Cargill
# McDonald's 
# SpaceX
# DoW

# Don't have large enough training data and am on time constraint now that these RAG algorithms didn't work out
# I didn't have the CPU/GPU resources to run efficiently, but can calibrate a deterministic pipeline/model with these examples